In [4]:
import os
# THE WINDOWS ENVIRONMENT FIXES
os.environ['HADOOP_HOME'] = r'C:\hadoop'
os.environ['PATH'] += os.pathsep + r'C:\hadoop\bin'
os.environ['PYSPARK_PYTHON'] = "python"
os.environ['PYSPARK_DRIVER_PYTHON'] = "python"

from pyspark.sql import SparkSession
from pyspark.ml import PipelineModel
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Mazra3atiAPI") \
    .getOrCreate()

model = PipelineModel.load(r"D:\Projects\mazra3ati\models\gbt_actual_yield_model")
print("Model loaded successfully")

Model loaded successfully


In [5]:
data = spark.read.csv(r"D:\Projects\mazra3ati\data\processed\mazra3ti_dataset.csv", header=True, inferSchema=True)
row = data.first()
print("First row of the dataset:")
print(row)
#prediction

data = data.withColumn(
    "demand_level_ord",
    F.when(F.col("demand_level") == "Low", 0)
     .when(F.col("demand_level") == "Medium", 1)
     .when(F.col("demand_level") == "High", 2)
     .otherwise(None)
)

data = (
    data
    .withColumn("planting_year", F.year(F.to_date("planting_date")))
    .withColumn("planting_month", F.month(F.to_date("planting_date")))
    .withColumn("planting_day", F.dayofmonth(F.to_date("planting_date")))
    .withColumn("sale_year", F.year(F.to_date("sale_date")))
    .withColumn("sale_month", F.month(F.to_date("sale_date")))
    .withColumn("sale_day", F.dayofmonth(F.to_date("sale_date")))
)


try:
    print("Columns ready for model:", data.columns)
    predictions = model.transform(data)
    predictions.select("record_id", "prediction").show()
except Exception as e:
    print("\n--- ERROR DURING INFERENCE ---")
    print(e)
    


First row of the dataset:
Row(record_id=1, crop_name='Peach', planting_date=datetime.date(2023, 3, 1), harvest_date=datetime.date(2023, 7, 24), season='Spring', area_size=109.56, quantity_sold=181.17, unit_price=3.984, total_sales=721.86, total_expenses=346.1, fertilizer_used=34.5, water_used=784.22, actual_yield=202.24, profit=375.76, sale_date=datetime.date(2023, 8, 20), demand_level='Medium', market_location='Maan', farm_location='Ajloun', region_type='Rural', distance_to_market=36.65, transport_cost=2.84, customer_type='Trader', market_price=3.899, supply_level=247.32, temperature=21.2, humidity=64.0, rainfall=13.2, wind_speed=8.2, pest_indicator=False)
Columns ready for model: ['record_id', 'crop_name', 'planting_date', 'harvest_date', 'season', 'area_size', 'quantity_sold', 'unit_price', 'total_sales', 'total_expenses', 'fertilizer_used', 'water_used', 'actual_yield', 'profit', 'sale_date', 'demand_level', 'market_location', 'farm_location', 'region_type', 'distance_to_market', '